# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets, fields and columns. Each entity is referenced by its `@id`.
record_sets = [r for r in metadata.record_sets]
print("Available record sets and their fields:\n")
for rs in record_sets:
    print(f"Record Set: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print("  Fields and columns:")
    for field in rs.fields:
        print(f"    Field: {field.name} (@id: {field.id})")
        if hasattr(field, "columns"):
            for col in field.columns:
                print(f"      Column: {col.name} (@id: {col.id})")
    print('---')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# We'll use all record sets listed above. You may modify this list as needed.
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set {record_set_id}")

# Display columns and a preview of the first record set
if record_set_ids:
    main_rs = record_set_ids[0]
    print(f"Columns in {main_rs}:")
    print(dataframes[main_rs].columns.tolist())
    dataframes[main_rs].head()
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations such as removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA on the first record set (modify as needed):

selected_rs_id = record_set_ids[0] if record_set_ids else None

if selected_rs_id and not dataframes[selected_rs_id].empty:
    df = dataframes[selected_rs_id]
    # Identify numeric fields (columns)
    numeric_fields = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]  # Just as an example
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field if exists
        group_fields = df.select_dtypes(include=['object']).columns.tolist()
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (group mean for numeric fields):")
            print(grouped_df.head())
    else:
        print(f"No numeric fields found to analyze in {selected_rs_id}.")
else:
    print("No records available to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id and not dataframes[selected_rs_id].empty:
    df = dataframes[selected_rs_id]
    numeric_fields = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_fields:
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_fields[0]], bins=30, kde=True)
        plt.title(f"Distribution of {numeric_fields[0]} in {selected_rs_id}")
        plt.xlabel(numeric_fields[0])
        plt.ylabel('Frequency')
        plt.show()

        # If two or more numeric fields, do a scatter plot
        if len(numeric_fields) > 1:
            plt.figure(figsize=(8, 5))
            sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
            plt.title(f"Scatter plot of {numeric_fields[0]} vs {numeric_fields[1]}")
            plt.xlabel(numeric_fields[0])
            plt.ylabel(numeric_fields[1])
            plt.show()
    else:
        print(f"No numeric fields available for visualization in {selected_rs_id}.")
else:
    print("No data available to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we loaded the Mass Spectrometry Analysis dataset using `mlcroissant`, explored record sets and fields by their `@id`, loaded tabular data for analysis, and performed basic exploratory analysis and visualization. Use these steps to further analyze the dataset for your own research questions. For more advanced analysis, see the official [`mlcroissant`](https://github.com/mlcommons/croissant) documentation.*